In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw/oulad")

print(f"Dataset directory: {DATA_DIR.resolve()}")

Dataset directory: /Users/fahad/Documents/MY#Documents/FAHAD ALI/Uni Wien/CLAF/data/raw/oulad


In [2]:
files = sorted(DATA_DIR.glob("*.csv"))

print(f"Number of CSV files: {len(files)}")
print()

for file in files:
    size_mb = file.stat().st_size / (1024 * 1024)
    print(f"{file.name:30} {size_mb:10.2f} MB")

Number of CSV files: 7

assessments.csv                      0.01 MB
courses.csv                          0.00 MB
studentAssessment.csv                5.43 MB
studentInfo.csv                      3.30 MB
studentRegistration.csv              1.06 MB
studentVle.csv                     432.81 MB
vle.csv                              0.25 MB


In [3]:
#Create a dataset inventory
inventory = []

for file in files:
    inventory.append({
        "file_name": file.name,
        "size_mb": round(file.stat().st_size / (1024 * 1024), 2)
    })

inventory_df = pd.DataFrame(inventory)

inventory_df

,file_name,size_mb
0,assessments.csv,0.01
1,courses.csv,0.00
2,studentAssessment.csv,5.43
3,studentInfo.csv,3.30
4,studentRegistration.csv,1.06
5,studentVle.csv,432.81
6,vle.csv,0.25


In [4]:
#Count rows

row_counts = []

for file in files:
    df = pd.read_csv(file)
    
    row_counts.append({
        "file_name": file.name,
        "rows": len(df),
        "columns": len(df.columns)
    })

row_counts_df = pd.DataFrame(row_counts)

row_counts_df

,file_name,rows,columns
0,assessments.csv,206,6
1,courses.csv,22,3
2,studentAssessment.csv,173912,5
3,studentInfo.csv,32593,12
4,studentRegistration.csv,32593,5
5,studentVle.csv,10655280,6
6,vle.csv,6364,6


In [5]:
#Inspect the schemas
schemas = {}

for file in files:
    sample = pd.read_csv(file, nrows=5)
    schemas[file.name] = list(sample.columns)

for file_name, columns in schemas.items():
    print(f"\n{file_name}")
    print("-" * len(file_name))
    
    for column in columns:
        print(f"  - {column}")


assessments.csv
---------------
  - code_module
  - code_presentation
  - id_assessment
  - assessment_type
  - date
  - weight

courses.csv
-----------
  - code_module
  - code_presentation
  - module_presentation_length

studentAssessment.csv
---------------------
  - id_assessment
  - id_student
  - date_submitted
  - is_banked
  - score

studentInfo.csv
---------------
  - code_module
  - code_presentation
  - id_student
  - gender
  - region
  - highest_education
  - imd_band
  - age_band
  - num_of_prev_attempts
  - studied_credits
  - disability
  - final_result

studentRegistration.csv
-----------------------
  - code_module
  - code_presentation
  - id_student
  - date_registration
  - date_unregistration

studentVle.csv
--------------
  - code_module
  - code_presentation
  - id_student
  - id_site
  - date
  - sum_click

vle.csv
-------
  - id_site
  - code_module
  - code_presentation
  - activity_type
  - week_from
  - week_to


In [6]:
#Load the small dimension/reference tables

student_info = pd.read_csv(DATA_DIR / "studentInfo.csv")
student_registration = pd.read_csv(DATA_DIR / "studentRegistration.csv")
assessments = pd.read_csv(DATA_DIR / "assessments.csv")
courses = pd.read_csv(DATA_DIR / "courses.csv")
vle = pd.read_csv(DATA_DIR / "vle.csv")
student_assessment = pd.read_csv(DATA_DIR / "studentAssessment.csv")

print("Small/reference tables loaded successfully.")

Small/reference tables loaded successfully.


In [7]:
#Check primary-key candidates

print("Unique students:", student_info["id_student"].nunique())
print("StudentInfo rows:", len(student_info))

print("\nUnique assessments:", assessments["id_assessment"].nunique())
print("Assessment rows:", len(assessments))

print("\nUnique VLE sites:", vle["id_site"].nunique())
print("VLE rows:", len(vle))

print("\nUnique course presentations:", 
      courses[["code_module", "code_presentation"]].drop_duplicates().shape[0])

print("Course rows:", len(courses))

Unique students: 28785
StudentInfo rows: 32593

Unique assessments: 206
Assessment rows: 206

Unique VLE sites: 6364
VLE rows: 6364

Unique course presentations: 22
Course rows: 22


In [8]:
#Check duplicate identifiers

print("Duplicate student IDs:",
      student_info["id_student"].duplicated().sum())

print("Duplicate assessment IDs:",
      assessments["id_assessment"].duplicated().sum())

print("Duplicate VLE site IDs:",
      vle["id_site"].duplicated().sum())

Duplicate student IDs: 3808
Duplicate assessment IDs: 0
Duplicate VLE site IDs: 0


In [9]:
#Validate Student → Registration
info_students = set(student_info["id_student"])

registration_students = set(student_registration["id_student"])

missing_students = registration_students - info_students

print("Students in registration:", len(registration_students))
print("Students missing from studentInfo:", len(missing_students))

Students in registration: 28785
Students missing from studentInfo: 0


In [10]:
#Validate Student → Assessment
assessment_students = set(student_assessment["id_student"])

missing_assessment_students = assessment_students - info_students

print("Students in studentAssessment:", len(assessment_students))
print(
    "Assessment students missing from studentInfo:",
    len(missing_assessment_students)
)

Students in studentAssessment: 23369
Assessment students missing from studentInfo: 0


In [11]:
#Validate Assessment → Assessment Definition
assessment_ids = set(assessments["id_assessment"])

student_assessment_ids = set(student_assessment["id_assessment"])

missing_assessments = student_assessment_ids - assessment_ids

print("Assessment IDs in studentAssessment:",
      len(student_assessment_ids))

print("Assessment IDs missing from assessments:",
      len(missing_assessments))

Assessment IDs in studentAssessment: 188
Assessment IDs missing from assessments: 0


In [12]:
#Validate VLE Site → VLE Definition

vle_ids = set(vle["id_site"])

missing_vle_sites = set()

total_rows = 0

for chunk in pd.read_csv(
    DATA_DIR / "studentVle.csv",
    usecols=["id_site"],
    chunksize=100_000
):
    total_rows += len(chunk)

    chunk_ids = set(chunk["id_site"].dropna().unique())

    missing_vle_sites.update(chunk_ids - vle_ids)

print("studentVle rows inspected:", total_rows)
print("Unique VLE sites referenced:", 
      total_rows)  # temporary; we'll calculate properly next
print("VLE sites missing from vle:", len(missing_vle_sites))

studentVle rows inspected: 10655280
Unique VLE sites referenced: 10655280
VLE sites missing from vle: 0


In [13]:
#Validate module/presentation relationships
#This is particularly important forProgram → Module → Cohort concept.
print("Courses:")
display(courses)

print("\nStudentInfo module/presentation combinations:")
display(
    student_info[
        ["code_module", "code_presentation"]
    ].drop_duplicates().sort_values(
        ["code_module", "code_presentation"]
    )
)

Courses:


,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268
3,BBB,2014J,262
4,BBB,2013B,240
5,BBB,2014B,234
6,CCC,2014J,269
7,CCC,2014B,241
8,DDD,2013J,261
9,DDD,2014J,262



StudentInfo module/presentation combinations:


,code_module,code_presentation
0,AAA,2013J
383,AAA,2014J
748,BBB,2013B
2515,BBB,2013J
4752,BBB,2014B
6365,BBB,2014J
8657,CCC,2014B
10593,CCC,2014J
13091,DDD,2013B
14394,DDD,2013J


In [14]:
#cohort and module analysis
#This is very relevant to earlier point about cohort and module analysis.
course_keys = set(
    zip(
        courses["code_module"],
        courses["code_presentation"]
    )
)

student_keys = set(
    zip(
        student_info["code_module"],
        student_info["code_presentation"]
    )
)

missing_course_keys = student_keys - course_keys

print(
    "Module/presentation combinations in studentInfo:",
    len(student_keys)
)

print(
    "Combinations missing from courses:",
    len(missing_course_keys)
)


Module/presentation combinations in studentInfo: 22
Combinations missing from courses: 0


In [15]:
unique_vle_sites = set()

for chunk in pd.read_csv(
    DATA_DIR / "studentVle.csv",
    usecols=["id_site"],
    chunksize=100_000
):
    unique_vle_sites.update(
        chunk["id_site"].dropna().unique()
    )

print("Unique VLE sites referenced:", len(unique_vle_sites))
print("VLE sites defined in vle:", len(vle_ids))
print("Missing VLE sites:", len(unique_vle_sites - vle_ids))

Unique VLE sites referenced: 6268
VLE sites defined in vle: 6364
Missing VLE sites: 0


In [16]:
# Inspect date ranges in the main temporal tables

print("Student VLE date range:")
student_vle_dates = pd.read_csv(
    DATA_DIR / "studentVle.csv",
    usecols=["date"]
)

print(student_vle_dates["date"].min())
print(student_vle_dates["date"].max())

print("\nAssessment date range:")
print(assessments["date"].min())
print(assessments["date"].max())

print("\nStudent assessment submission date range:")
print(student_assessment["date_submitted"].min())
print(student_assessment["date_submitted"].max())

Student VLE date range:
-25
269

Assessment date range:
12.0
261.0

Student assessment submission date range:
-11
608


In [19]:
activity_by_day = (
    student_vle_dates
    .groupby("date")
    .size()
    .reset_index(name="activity_records")
)

activity_by_day.head(20)


,date,activity_records
0,-25,4291
1,-24,3887
2,-23,2618
3,-22,1728
4,-21,1283
5,-20,1618
6,-19,1699
7,-18,57920
8,-17,32336
9,-16,31053


In [20]:
print("Number of activity days:", activity_by_day["date"].nunique())
print("Minimum day:", activity_by_day["date"].min())
print("Maximum day:", activity_by_day["date"].max())

Number of activity days: 295
Minimum day: -25
Maximum day: 269


In [22]:
#Presentation Duration
course_duration = courses[
    ["code_module", "code_presentation", "module_presentation_length"]
].copy()

course_duration

,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268
3,BBB,2014J,262
4,BBB,2013B,240
5,BBB,2014B,234
6,CCC,2014J,269
7,CCC,2014B,241
8,DDD,2013J,261
9,DDD,2014J,262


In [23]:
print(
    course_duration["module_presentation_length"].describe()
)


count     22.000000
mean     255.545455
std       13.654677
min      234.000000
25%      241.000000
50%      261.500000
75%      268.000000
max      269.000000
Name: module_presentation_length, dtype: float64


In [25]:
#Activity coverage by module/presentation
activity_coverage = []

for chunk in pd.read_csv(
    DATA_DIR / "studentVle.csv",
    usecols=[
        "code_module",
        "code_presentation",
        "date"
    ],
    chunksize=100_000
):
    
    grouped = (
        chunk
        .groupby(
            ["code_module", "code_presentation"]
        )["date"]
        .agg(
            min_activity_day="min",
            max_activity_day="max",
            unique_activity_days="nunique"
        )
        .reset_index()
    )
    
    activity_coverage.append(grouped)

activity_coverage_df = pd.concat(
    activity_coverage,
    ignore_index=True
)

activity_coverage_df

,code_module,code_presentation,min_activity_day,max_activity_day,unique_activity_days
0,AAA,2013J,-10,116,127
1,AAA,2013J,116,268,153
2,AAA,2014J,-24,3,28
3,AAA,2014J,3,156,154
4,AAA,2014J,156,269,114
...,...,...,...,...,...
123,GGG,2013J,219,261,43
124,GGG,2014B,-8,167,176
125,GGG,2014B,167,241,73
126,GGG,2014J,-16,130,147


In [26]:
activity_coverage_final = (
    activity_coverage_df
    .groupby(
        ["code_module", "code_presentation"]
    )
    .agg(
        min_activity_day=("min_activity_day", "min"),
        max_activity_day=("max_activity_day", "max"),
        unique_activity_days=("unique_activity_days", "sum")
    )
    .reset_index()
)

activity_coverage_final

,code_module,code_presentation,min_activity_day,max_activity_day,unique_activity_days
0,AAA,2013J,-10,268,280
1,AAA,2014J,-24,269,296
2,BBB,2013B,-9,240,254
3,BBB,2013J,-23,268,285
4,BBB,2014B,-9,234,244
5,BBB,2014J,-9,262,277
6,CCC,2014B,-18,241,263
7,CCC,2014J,-18,269,295
8,DDD,2013B,-16,240,262
9,DDD,2013J,-18,261,287


In [27]:
activity_coverage_final = activity_coverage_final.merge(
    course_duration,
    on=["code_module", "code_presentation"],
    how="left"
)

activity_coverage_final

,code_module,code_presentation,min_activity_day,max_activity_day,unique_activity_days,module_presentation_length
0,AAA,2013J,-10,268,280,268
1,AAA,2014J,-24,269,296,269
2,BBB,2013B,-9,240,254,240
3,BBB,2013J,-23,268,285,268
4,BBB,2014B,-9,234,244,234
5,BBB,2014J,-9,262,277,262
6,CCC,2014B,-18,241,263,241
7,CCC,2014J,-18,269,295,269
8,DDD,2013B,-16,240,262,240
9,DDD,2013J,-18,261,287,261


In [28]:
coverage_summary = activity_coverage_final.copy()

coverage_summary["observed_span_days"] = (
    coverage_summary["max_activity_day"]
    - coverage_summary["min_activity_day"]
    + 1
)

coverage_summary["coverage_ratio"] = (
    coverage_summary["unique_activity_days"]
    / coverage_summary["module_presentation_length"]
)

coverage_summary[
    [
        "code_module",
        "code_presentation",
        "module_presentation_length",
        "min_activity_day",
        "max_activity_day",
        "unique_activity_days",
        "observed_span_days",
        "coverage_ratio"
    ]
]

,code_module,code_presentation,module_presentation_length,min_activity_day,max_activity_day,unique_activity_days,observed_span_days,coverage_ratio
0,AAA,2013J,268,-10,268,280,279,1.044776
1,AAA,2014J,269,-24,269,296,294,1.100372
2,BBB,2013B,240,-9,240,254,250,1.058333
3,BBB,2013J,268,-23,268,285,292,1.063433
4,BBB,2014B,234,-9,234,244,244,1.042735
5,BBB,2014J,262,-9,262,277,272,1.057252
6,CCC,2014B,241,-18,241,263,260,1.091286
7,CCC,2014J,269,-18,269,295,288,1.096654
8,DDD,2013B,240,-16,240,262,257,1.091667
9,DDD,2013J,261,-18,261,287,280,1.099617


In [29]:
print("Minimum coverage ratio:",
      coverage_summary["coverage_ratio"].min())

print("Maximum coverage ratio:",
      coverage_summary["coverage_ratio"].max())

print("Mean coverage ratio:",
      coverage_summary["coverage_ratio"].mean())

print("\nPresentations below 80% coverage:")
display(
    coverage_summary[
        coverage_summary["coverage_ratio"] < 0.80
    ]
)

Minimum coverage ratio: 1.033195020746888
Maximum coverage ratio: 1.1208333333333333
Mean coverage ratio: 1.0816423155986485

Presentations below 80% coverage:


,code_module,code_presentation,min_activity_day,max_activity_day,unique_activity_days,module_presentation_length,observed_span_days,coverage_ratio
